# Step 5.ii — ChatGPT and Claude function extraction (via APIs)

Same task as `openllm_function_extraction.ipynb`, but driven by the OpenAI and Anthropic APIs instead of the chat web UIs. Plug your two keys into the cell below and run all.

**Prompt is byte-identical to step 5.i** so the comparison against the open-source models stays apples-to-apples.

Outputs (under `/content/work/results/`, then zipped and downloaded):
* `chatgpt_functions.txt`, `chatgpt_per_file.csv`, `chatgpt_summary.txt`
* `claude_functions.txt`, `claude_per_file.csv`, `claude_summary.txt`

Notes:
* These APIs have huge context windows, so each `.c` file is sent in a single call (no chunking like the small open-source models needed).
* Per-file checkpoint CSV is written after every file, so re-running resumes where it left off.

## 1. Install SDKs

In [ ]:
!pip -q install "openai>=1.50" "anthropic>=0.40"

## 2. API keys + model selection

Paste your keys when prompted (they're not echoed and not saved to the notebook). Change the model names if you want a different tier.

In [ ]:
import getpass, os

os.environ['OPENAI_API_KEY']    = os.environ.get('OPENAI_API_KEY')    or getpass.getpass('OpenAI API key: ')
os.environ['ANTHROPIC_API_KEY'] = os.environ.get('ANTHROPIC_API_KEY') or getpass.getpass('Anthropic API key: ')

# Pick which to run. Set to None or [] to skip one.
OPENAI_MODEL    = 'gpt-5'              # e.g. 'gpt-5', 'gpt-4o'
ANTHROPIC_MODEL = 'claude-sonnet-4-6'  # e.g. 'claude-sonnet-4-6', 'claude-opus-4-7'

MODELS_TO_RUN = ['chatgpt', 'claude']  # subset of ['chatgpt', 'claude']

MAX_OUTPUT_TOKENS = 4096
REQUEST_TIMEOUT_S = 120

## 3. Download and extract coreutils-9.1

In [ ]:
import os, subprocess, pathlib

WORK = pathlib.Path('/content/work')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

if not (WORK / 'coreutils-9.1').exists():
    subprocess.check_call(['wget', '-q', 'https://ftp.gnu.org/gnu/coreutils/coreutils-9.1.tar.xz'])
    subprocess.check_call(['tar', '-xf', 'coreutils-9.1.tar.xz'])

SRC_DIR = WORK / 'coreutils-9.1' / 'src'
C_FILES = sorted(SRC_DIR.glob('*.c'))
print(f'Found {len(C_FILES)} .c files in {SRC_DIR}')
print('First 5:', [p.name for p in C_FILES[:5]])

## 4. Prompt and parsing helpers

Same `PROMPT_TEMPLATE` and `parse_names` as the open-source notebook, so the comparison is fair.

In [ ]:
import re

PROMPT_TEMPLATE = '''You are a C source-code analyzer.

Below is a C source file from GNU coreutils. List every function that is DEFINED in this file (not merely declared, included, or called).

Rules:
- Output ONLY function names, one per line.
- Do not output return types, parameters, parentheses, comments, prose, or markdown.
- Do not include macros, typedefs, or struct names.
- Do not include functions that are only called or only declared.
- If a function is static, still include its name.

File: {filename}

```c
{code}
```

Function names (one per line):'''

C_IDENT = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')
_LEAD = re.compile(r'^[\\s\\-\\*\u2022`]+|^\\d+[\\.\\)]\\s*')
_KEYWORDS = {'if','else','for','while','switch','case','return','static','void',
             'int','char','struct','typedef','sizeof','goto','break','continue',
             'default','do','enum','union','const','extern','register','signed',
             'unsigned','volatile','inline','restrict'}

def parse_names(text: str) -> list:
    out = []
    for line in text.splitlines():
        s = line.strip()
        prev = None
        while s and s != prev:
            prev = s
            s = _LEAD.sub('', s).strip().strip('`*_')
        s = re.sub(r'\\(.*$', '', s).strip().rstrip(';, ')
        if not s or not C_IDENT.match(s) or s in _KEYWORDS:
            continue
        out.append(s)
    return out

## 5. API client wrappers

Greedy / deterministic where the API allows it. Both wrappers retry on transient errors with simple exponential backoff.

In [ ]:
import time
from openai import OpenAI
from anthropic import Anthropic

_oai = OpenAI(timeout=REQUEST_TIMEOUT_S)
_ant = Anthropic(timeout=REQUEST_TIMEOUT_S)

def _retry(fn, *, attempts=4):
    delay = 2.0
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            if i == attempts - 1:
                raise
            print(f'    retry {i+1}/{attempts-1} after {type(e).__name__}: {e}')
            time.sleep(delay)
            delay *= 2

def chatgpt_generate(prompt: str) -> str:
    def call():
        resp = _oai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_completion_tokens=MAX_OUTPUT_TOKENS,
        )
        return resp.choices[0].message.content or ''
    return _retry(call)

def claude_generate(prompt: str) -> str:
    def call():
        resp = _ant.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=MAX_OUTPUT_TOKENS,
            messages=[{'role': 'user', 'content': prompt}],
        )
        # Concatenate any text blocks in the response.
        return ''.join(b.text for b in resp.content if getattr(b, 'type', None) == 'text')
    return _retry(call)

GENERATORS = {'chatgpt': chatgpt_generate, 'claude': claude_generate}
MODEL_IDS  = {'chatgpt': OPENAI_MODEL,    'claude': ANTHROPIC_MODEL}

## 6. Run each model over every `.c` file

In [ ]:
import pandas as pd

RESULTS = WORK / 'results'
RESULTS.mkdir(exist_ok=True)

summary_rows = []

for label in MODELS_TO_RUN:
    if label not in GENERATORS:
        print(f'!! unknown model label: {label}, skipping')
        continue
    gen = GENERATORS[label]
    print(f'\n=== {label} ({MODEL_IDS[label]}) ===')

    ckpt_per_file = RESULTS / f'{label}_per_file.csv'
    if ckpt_per_file.exists():
        done_df = pd.read_csv(ckpt_per_file)
        per_file_rows = done_df.to_dict('records')
        done_files = set(done_df['file'].tolist())
        all_names = set()
        for s in done_df['functions'].fillna(''):
            all_names.update(n for n in s.split(';') if n)
        print(f'  resume: {len(done_files)} files done, {len(all_names)} unique names so far')
    else:
        per_file_rows, done_files, all_names = [], set(), set()

    todo = [p for p in C_FILES if p.name not in done_files]
    if not todo:
        print('  nothing to do, all files already in checkpoint.')
        # still record summary
        summary_rows.append({'model': label, 'model_id': MODEL_IDS[label],
                             'unique_functions': len(all_names),
                             'files_processed': len(C_FILES),
                             'wall_clock_sec': 0.0, 'errors': 0})
        continue

    t0 = time.time()
    error_count = 0
    total_definitions = sum(int(r.get('n_functions', 0)) for r in per_file_rows)

    for i, path in enumerate(todo, 1):
        src = path.read_text(errors='replace')
        prompt = PROMPT_TEMPLATE.format(filename=path.name, code=src)
        try:
            resp = gen(prompt)
        except Exception as e:
            error_count += 1
            print(f'  ! {path.name}: {type(e).__name__}: {e}')
            resp = ''

        names_for_file = sorted(set(parse_names(resp)))
        all_names.update(names_for_file)
        total_definitions += len(names_for_file)
        per_file_rows.append({
            'file': path.name,
            'n_functions': len(names_for_file),
            'functions': ';'.join(names_for_file),
        })
        pd.DataFrame(per_file_rows).to_csv(ckpt_per_file, index=False)

        if i % 10 == 0 or i == len(todo):
            print(f'  [{i}/{len(todo)}] {path.name:25s}  unique-so-far={len(all_names)}')

    elapsed = time.time() - t0

    # Final outputs.
    (RESULTS / f'{label}_functions.txt').write_text('\n'.join(sorted(all_names)) + '\n')
    (RESULTS / f'{label}_summary.txt').write_text(
        f'MODEL: {MODEL_IDS[label]}\n'
        f'TOTAL_DEFINITIONS: {total_definitions}\n'
        f'UNIQUE_NAMES: {len(all_names)}\n'
        f'FILES_PROCESSED: {len(C_FILES)}\n'
        f'WALL_CLOCK_SEC: {elapsed:.1f}\n'
        f'ERRORS: {error_count}\n'
    )
    summary_rows.append({'model': label, 'model_id': MODEL_IDS[label],
                         'unique_functions': len(all_names),
                         'files_processed': len(C_FILES),
                         'wall_clock_sec': round(elapsed, 1),
                         'errors': error_count})
    print(f'  -> {len(all_names)} unique function names in {elapsed:.1f}s ({error_count} errors)')

summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS / 'api_summary.csv', index=False)
print('\nFinal summary:')
print(summary.to_string(index=False))

## 7. Zip and download results

In [ ]:
import shutil
shutil.make_archive('/content/api_results', 'zip', RESULTS)
from google.colab import files
files.download('/content/api_results.zip')